In [7]:

import argparse
import math
import sys
from pathlib import Path
 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
 
import torch
import torch.nn as nn
from torchvision import models, transforms
from torchvision.transforms import v2
from PIL import Image

In [8]:
BG      = "#0d0d0f"
SURFACE = "#16161a"
ACCENT  = "#7f5af0"
TEXT    = "#fffffe"
SUBTLE  = "#72757e"
 
plt.rcParams.update({
    "figure.facecolor":  BG,
    "axes.facecolor":    SURFACE,
    "axes.edgecolor":    SUBTLE,
    "axes.labelcolor":   TEXT,
    "xtick.color":       SUBTLE,
    "ytick.color":       SUBTLE,
    "text.color":        TEXT,
    "font.family":       "monospace",
})
 

In [9]:
class AlexNet(nn.Module):
    def __init__(self, dropout: float = 0.5):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 96, kernel_size=11, stride=4),
            nn.BatchNorm2d(96),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),

            # Block 2
            nn.Conv2d(96, 256, kernel_size=5, padding=2),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),

            # Block 3
            nn.Conv2d(256, 384, kernel_size=3, padding=1),
            nn.BatchNorm2d(384),
            nn.ReLU(inplace=True),

            # Block 4
            nn.Conv2d(384, 384, kernel_size=3, padding=1),
            nn.BatchNorm2d(384),
            nn.ReLU(inplace=True),

            # Block 5
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )

        self.avgpool = nn.AdaptiveAvgPool2d((6, 6))

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),

            nn.Dropout(dropout),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),

            nn.Linear(4096, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

In [10]:
def load_alexnet(weight_path):
    model = AlexNet()
    model.load_state_dict(torch.load(weight_path, map_location="cpu"))
    model.eval()
    return model

In [11]:
def get_named_conv_layers(model):
    """Return (name, module) pairs for every Conv2d in the model."""
    return [(name, m) for name, m in model.named_modules()
            if isinstance(m, nn.Conv2d)]

In [12]:
# model = load_alexnet("../best_alexnet_spurious_0.5.pt")
# print(get_named_conv_layers(model))

In [13]:
def register_activation_hooks(model):
    """
    Attach forward hooks to every Conv2d and ReLU so we can capture
    the feature maps that flow through the network.
    Returns (activations dict, list of hook handles).
    """
    activations = {}
    handles = []
 
    def make_hook(name):
        def hook(module, inp, out):
            activations[name] = out.detach()
        return hook
 
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.ReLU, nn.MaxPool2d, nn.AdaptiveAvgPool2d)):
            handles.append(module.register_forward_hook(make_hook(name)))
 
    return activations, handles

In [14]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
 
preprocess = transforms.Compose([
    v2.Grayscale(num_output_channels=1),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
])
 

In [15]:
def load_image(path=None):
    if path:
        pil = Image.open(path).convert("RGB")
    else:
        # Structured noise: more interesting than uniform random
        rng = np.random.default_rng(42)
        arr = rng.integers(0, 256, (224, 224, 3), dtype=np.uint8)
        # Add some low-freq structure so filters have something to respond to
        for c in range(3):
            arr[:, :, c] = (arr[:, :, c] * 0.4 +
                            np.tile(rng.integers(0, 256, (28, 28)),
                                    (8, 8))[:224, :224] * 0.6).astype(np.uint8)
        pil = Image.fromarray(arr)
 
    tensor = preprocess(pil).unsqueeze(0)
    return pil, tensor
 

In [16]:
def denormalize(tensor):
    """Convert a normalized [C,H,W] tensor back to a displayable numpy array."""
    t = tensor.clone().numpy()
    # for c, (m, s) in enumerate(zip(IMAGENET_MEAN, IMAGENET_STD)):
    #     t[c] = t[c] * s + m
    return np.clip(t.transpose(1, 2, 0), 0, 1)
 

In [28]:
def visualize_filters(model, save_path="alexnet_filters.png"):
    """
    Visualize the learned convolutional filter weights for every Conv2d layer.
    Each tile = one filter (shown as its RGB channels or grayscale mean).
    """
    conv_layers = get_named_conv_layers(model)
    n_layers = len(conv_layers)
 
    fig = plt.figure(figsize=(20, 4 * n_layers))
    fig.suptitle("AlexNet · Learned Filter Weights (per Conv layer)",
                 fontsize=16, fontweight="bold", color=TEXT, y=1.01)
 
    for row_idx, (name, layer) in enumerate(conv_layers):
        weights = layer.weight.data.clone()        # [out_ch, in_ch, kH, kW]
        n_filters = weights.shape[0]
 
        # Normalise each filter to [0,1] for display
        w_min, w_max = weights.min(), weights.max()
        weights = (weights - w_min) / (w_max - w_min + 1e-8)
 
        n_cols = min(n_filters, 32)
        n_rows_inner = math.ceil(n_filters / n_cols)
 
        # Outer subplot for the layer label
        ax_label = fig.add_subplot(n_layers, 1, row_idx + 1)
        ax_label.axis("off")
        ax_label.set_title(
            f"Layer: {name}   |   shape: {list(layer.weight.shape)}   "
            f"|   stride: {layer.stride}   pad: {layer.padding}",
            loc="left", color=ACCENT, fontsize=11, pad=8,
        )
 
        # Inner grid of filter tiles
        gs_inner = gridspec.GridSpecFromSubplotSpec(
            n_rows_inner, n_cols, subplot_spec=ax_label.get_subplotspec(),
            hspace=0.05, wspace=0.05,
        )
 
        for fi in range(n_filters):
            ax = fig.add_subplot(gs_inner[fi // n_cols, fi % n_cols])
            f = weights[fi]                          # [in_ch, kH, kW]
            if f.shape[0] == 3:
                # Only the first layer of a 3-channel input model
                img = f[:3].permute(1, 2, 0).numpy()
                img = (img - img.min()) / (img.max() - img.min() + 1e-8)
                ax.imshow(img, interpolation="nearest")
            else:
                # Single-channel input OR deeper layers with many input channels
                img = f.mean(0).numpy()
                img = (img - img.min()) / (img.max() - img.min() + 1e-8)
                ax.imshow(img, cmap="gray", interpolation="nearest")          # grayscale mean
            ax.axis("off")
 
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight",
                facecolor=BG, edgecolor="none")
    plt.close(fig)
    print(f"  ✓  Filter weights  → {save_path}")
 

In [18]:
def visualize_activations(model, image_tensor, pil_image,
                          save_path="alexnet_activations.png"):
    """
    Run a forward pass and display the feature maps produced at each
    Conv2d + ReLU + Pool layer (up to 32 channels each).
    """
    activations, handles = register_activation_hooks(model)
 
    with torch.no_grad():
        output = model(image_tensor)
    # pred_class = output.argmax(dim=1).item()
 
    for h in handles:
        h.remove()
 
    # Keep only the layers we want to show, in order
    interesting = {k: v for k, v in activations.items()
                   if any(t in k for t in ("features.", "avgpool"))}
 
    n_layers = len(interesting)
    fig = plt.figure(figsize=(22, 4.5 * n_layers))
    fig.suptitle(
        f"AlexNet · Activation Maps ",
        fontsize=15, fontweight="bold", color=TEXT, y=1.005,
    )
 
    for row_idx, (name, act) in enumerate(interesting.items()):
        act_np = act[0].numpy()                   # [C, H, W]
        n_ch   = act_np.shape[0]
        n_cols = min(n_ch, 32)
        n_rows_inner = math.ceil(min(n_ch, 32) / n_cols)
 
        ax_label = fig.add_subplot(n_layers, 1, row_idx + 1)
        ax_label.axis("off")
        ax_label.set_title(
            f"Layer: {name}   |   output shape: {list(act.shape[1:])}",
            loc="left", color=ACCENT, fontsize=11, pad=8,
        )
 
        gs_inner = gridspec.GridSpecFromSubplotSpec(
            n_rows_inner, n_cols + 1,               # +1 for input thumb
            subplot_spec=ax_label.get_subplotspec(),
            hspace=0.05, wspace=0.05,
        )
 
        # Input thumbnail on the left
        ax_in = fig.add_subplot(gs_inner[:, 0])
        ax_in.imshow(pil_image.resize((56, 56)))
        ax_in.set_title("input", fontsize=7, color=SUBTLE, pad=2)
        ax_in.axis("off")
 
        for ci in range(min(n_ch, 32)):
            ax = fig.add_subplot(gs_inner[ci // n_cols, (ci % n_cols) + 1])
            ch = act_np[ci]
            ax.imshow(ch, cmap="inferno", interpolation="nearest")
            ax.axis("off")
 
    plt.tight_layout()
    fig.savefig(save_path, dpi=130, bbox_inches="tight",
                facecolor=BG, edgecolor="none")
    plt.close(fig)
    print(f"Activation maps → {save_path}")
 
 

In [19]:

def visualize_saliency(model, image_tensor, pil_image,
                       save_path="alexnet_saliency.png"):
    """
    Vanilla gradient saliency: backprop from the top predicted class score
    to the input image. Bright pixels = pixels the model is most sensitive to.
    Also shows Guided Backprop for sharper spatial attribution.
    """
    # ── Vanilla saliency ──────────────────────────────────────────────────────
    inp = image_tensor.clone().requires_grad_(True)
    output = model(inp)
    pred_class = output.argmax(dim=1).item()
    # Just takes the logits, argmax will always be 0
    score = output[0, pred_class]
    model.zero_grad()
    score.backward()
 
    saliency = inp.grad.data.abs()[0]              # [3, H, W]
    saliency_max, _ = saliency.max(dim=0)          # max over channels
    saliency_np = saliency_max.numpy()
 
    # ── Guided Backprop ───────────────────────────────────────────────────────
    # Replace ReLU backward with a guided version (only pass positive gradients
    # that come from positive activations).
    saved_relu_bwd = {}
 
    def guided_hook_factory(module):
        def backward_hook(module, grad_in, grad_out):
            return (torch.clamp(grad_in[0], min=0),)
        return module.register_backward_hook(backward_hook)
 
    guided_handles = []
    for m in model.modules():
        if isinstance(m, nn.ReLU):
            guided_handles.append(guided_hook_factory(m))
 
    inp_g = image_tensor.clone().requires_grad_(True)
    out_g = model(inp_g)
    model.zero_grad()
    out_g[0, pred_class].backward()
 
    guided_sal = inp_g.grad.data[0]               # [3, H, W]
    guided_sal = guided_sal - guided_sal.min()
    guided_sal = guided_sal / (guided_sal.max() + 1e-8)
    guided_sal_np = guided_sal.permute(1, 2, 0).numpy()
 
    for h in guided_handles:
        h.remove()
 
    # ── Plot ──────────────────────────────────────────────────────────────────
    input_display = denormalize(image_tensor[0])
 
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle(
        f"AlexNet · Gradient Saliency",
        fontsize=14, fontweight="bold", color=TEXT,
    )
    fig.patch.set_facecolor(BG)
 
    panels = [
        (input_display,          "Input Image",      "viridis",  False),
        (saliency_np,            "Vanilla Saliency", "hot",      True),
        (guided_sal_np,          "Guided Backprop",  "viridis",  False),
        (input_display * np.expand_dims(saliency_np / saliency_np.max(), -1),
                                 "Overlay",          "viridis",  False),
    ]
 
    for ax, (img, title, cmap, use_cmap) in zip(axes, panels):
        ax.set_facecolor(SURFACE)
        ax.imshow(img, cmap=cmap if use_cmap else None)
        ax.set_title(title, color=ACCENT, fontsize=12, pad=6)
        ax.axis("off")
        for spine in ax.spines.values():
            spine.set_edgecolor(SUBTLE)
 
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight",
                facecolor=BG, edgecolor="none")
    plt.close(fig)
    print(f"Gradient saliency → {save_path}")
 
 

In [20]:
def print_architecture(model):
    print("\n── AlexNet Architecture ──────────────────────────────────────")
    for name, m in model.named_modules():
        if isinstance(m, (nn.Conv2d, nn.Linear, nn.MaxPool2d,
                          nn.ReLU, nn.Dropout, nn.AdaptiveAvgPool2d)):
            indent = "  " * name.count(".")
            params = (f"  [{sum(p.numel() for p in m.parameters()):,} params]"
                      if list(m.parameters()) else "")
            print(f"  {indent}{name:30s}  {m}{params}")
    print()
 

In [29]:
outdir = Path("./")
outdir.mkdir(parents=True, exist_ok=True)

print("\nLoading pretrained AlexNet …")
model = load_alexnet("../model_pths/best_alexnet_clean.pt")
print_architecture(model)

print("Loading image …")
pil_image, image_tensor = load_image("../test_images/img5_star.png")

visualize_filters(model, save_path=str(outdir / "alexnet_filters.png"))

visualize_activations(model, image_tensor, pil_image, save_path=str(outdir / "alexnet_activations.png"))

# if mode in ("all", "saliency"):
visualize_saliency(model, image_tensor, pil_image, save_path=str(outdir / "alexnet_saliency.png"))

print("\nDone. Output files saved to:", outdir.resolve())


Loading pretrained AlexNet …

── AlexNet Architecture ──────────────────────────────────────
    features.0                      Conv2d(1, 96, kernel_size=(11, 11), stride=(4, 4))  [11,712 params]
    features.2                      ReLU(inplace=True)
    features.3                      MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    features.4                      Conv2d(96, 256, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))  [614,656 params]
    features.6                      ReLU(inplace=True)
    features.7                      MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    features.8                      Conv2d(256, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))  [885,120 params]
    features.10                     ReLU(inplace=True)
    features.11                     Conv2d(384, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))  [1,327,488 params]
    features.13                     ReLU(inplace